# 00 - Deep Mathematical Baseline & Fair Checking

This foundational notebook implements the exact 40/60 momentum formula and dynamic affection-rate banding to create perfectly contextual targets for classification. It ensures that all 12 subsequent machine learning models are evaluated on an identical, strictly verified foundation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the purely chronological splits from Phase 1
train_df = pd.read_csv('../../../data/processed/train.csv')
val_df = pd.read_csv('../../../data/processed/val.csv')

print(f"Train Shape: {train_df.shape} | Val Shape: {val_df.shape}")

Train Shape: (360, 48) | Val Shape: (77, 48)


### Step 1 & 2: Base Momentum and Affection Rates ($W$)
We calculate the 40/60 base momentum, and then rigorously scan the training dataset to calculate the mean positive/negative impact (Affection Rate) of every single binary categorical variable.

In [2]:
global_mean = train_df['Attendance_Percentage'].mean()
print(f"Global Average Attendance (Train): {global_mean:.2f}%\n")

# 1. Base Momentum Formula (40% Long-term, 60% Short-term)

# calculate_momentum removed by agent
train_df['Attendance_Class'], attendance_bins = pd.qcut(train_df['Attendance_Percentage'], q=3, labels=['Low', 'Medium', 'High'], retbins=True)
val_df['Attendance_Class'] = pd.cut(val_df['Attendance_Percentage'], bins=attendance_bins, labels=['Low', 'Medium', 'High'], include_lowest=True)

train_df.dropna(subset=['Attendance_Class'], inplace=True)
val_df.dropna(subset=['Attendance_Class'], inplace=True)



Global Average Attendance (Train): 76.28%



### Step 3: Expected Attendance ($E_i$) & Residual Calculation ($R_i$)
We sum the Base Momentum and all applicable Affection Rates for every single lecture to form the `Expected_Attendance`.
Then, we calculate the Residual (Actual - Expected).

In [3]:
# expected and residual removed by agent


### Step 4: Dynamic Tertile Banding
We use Quantiles on the Training Residuals to set the exact boundaries for High, Medium, and Low. This completely eliminates class imbalance.

In [4]:
print("--- Attendance Boundaries (Derived from Train) ---")
print(f"Low Band:    Attendance <  {attendance_bins[1]:.2f}%")
print(f"Medium Band: Attendance >= {attendance_bins[1]:.2f}% AND <= {attendance_bins[2]:.2f}%")
print(f"High Band:   Attendance >  {attendance_bins[2]:.2f}%")

print("\n--- Class Distribution (Train) ---")
print(train_df['Attendance_Class'].value_counts(normalize=True))



--- Attendance Boundaries (Derived from Train) ---
Low Band:    Attendance <  73.53%
Medium Band: Attendance >= 73.53% AND <= 79.90%
High Band:   Attendance >  79.90%

--- Class Distribution (Train) ---
Attendance_Class
Medium    0.352778
Low       0.338889
High      0.308333
Name: proportion, dtype: float64


### Step 5: Leakage Prevention (Applying to Validation Set)
We apply the exact same formula, exact same Affection Rates, and exact same Cutoffs to the Validation Set. The validation set is NEVER allowed to influence the weights.

In [5]:
# expected and residual removed by agent


### Step 6: Target Separation & Standard Scaling
We separate the targets (`y`) from the features (`X`), and apply a `StandardScaler` to ensure algorithms like SVM and KNN are judged fairly.

In [6]:
targets_to_drop = [
    'Attendance_Percentage', 'Attendance_Class', 'Expected_Attendance', 'Residual', 'Base_Momentum',
    'Students_Present', 'Total_Enrolled', 'Date', 'Start_Time', 'End_Time', 'Faculty_ID', 'Semester', 'Branch', 'Section', 'Classroom', 'Special_Event', 'Assignment_Due_Flag'
]

y_train_reg = train_df['Attendance_Percentage']
y_train_class = train_df['Attendance_Class']
X_train = train_df.drop(columns=[c for c in targets_to_drop if c in train_df.columns]).select_dtypes(exclude=['object', 'string'])

y_val_reg = val_df['Attendance_Percentage']
y_val_class = val_df['Attendance_Class']
X_val = val_df.drop(columns=[c for c in targets_to_drop if c in val_df.columns]).select_dtypes(exclude=['object', 'string'])

In [7]:

from sklearn.preprocessing import StandardScaler
import pandas as pd
import os

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)

out_dir = '../../../data/processed'
os.makedirs(out_dir, exist_ok=True)

X_train_scaled.to_csv(f"{out_dir}/X_train_scaled.csv", index=False)
X_val_scaled.to_csv(f"{out_dir}/X_val_scaled.csv", index=False)

y_train_class.to_csv(f"{out_dir}/y_train_class.csv", index=False)
y_val_class.to_csv(f"{out_dir}/y_val_class.csv", index=False)

y_train_reg.to_csv(f"{out_dir}/y_train_reg.csv", index=False)
y_val_reg.to_csv(f"{out_dir}/y_val_reg.csv", index=False)

print("Scaled and exported all matrices to data/processed/")


Scaled and exported all matrices to data/processed/
